# Analisador de Áudio do Google Drive

Este notebook foi criado pela [biodiversica](https://biodiversica.xyz) e lê gravações de áudio diretamente do seu **Google Drive** para executar um **Modelo de Classificação por Rede Neural** e detectar espécies específicas em cada gravação.

---

### O que este notebook faz (em ordem):
1. **Conecta ao seu Google Drive** para ler o áudio e salvar resultados
2. **Instala os softwares necessários** automaticamente
3. **Varre a pasta de áudio** para descobrir pontos de gravação e arquivos
4. **Carrega um modelo** do HuggingFace ou do seu Google Drive (ex.: BirdNET, Perch, personalizado)
5. **Analisa cada gravação** e salva os resultados como arquivos de texto no seu Drive

### Estrutura de pastas:
Você pode organizar seus arquivos de áudio de duas formas:

```
# Plana — todos os arquivos em uma pasta (ponto único)
MyDrive/audio/
  20250101_180000.wav
  20250101_183000.wav

# Subpastas — cada subpasta é um ponto de gravação separado
MyDrive/audio/
  PONTO_A/
    20250101_180000.wav
  PONTO_B/
    20250101_180000.wav
```

### Formato dos nomes de arquivo:
O notebook lê os timestamps dos nomes dos arquivos. Formatos suportados:
- **AudioMoth** (padrão): `YYYYMMDD_HHMMSS.wav` — ex.: `20250615_203000.wav`
- **AudioMoth com prefixo**: `PREFIXO_YYYYMMDD_HHMMSS.wav` — ex.: `SM4_20250615_203000.wav`
- **ISO datetime**: `YYYY-MM-DD_HH-MM-SS.wav` — ex.: `2025-06-15_20-30-00.wav`

Arquivos cujos nomes não correspondem a nenhum padrão ainda são analisados — as colunas baseadas em timestamp nos resultados ficarão em branco.

### Antes de começar:
- Você precisa de uma **conta Google** com Google Drive
- Você precisa de um **arquivo de modelo TFLite (`.tflite`) ou ONNX (`.onnx`)** e um **arquivo de rótulos** compatível (`.txt`, uma espécie por linha)
- Seus arquivos de áudio devem estar no Google Drive

### Como executar:
Execute cada célula **uma de cada vez**, de cima para baixo. Clique no botão ▶ à esquerda de cada célula ou pressione `Shift + Enter`.

> **Novo em notebooks?** Uma célula com `[ ]` à esquerda ainda não foi executada. Após executar, aparece `[1]`, `[2]`, etc. Se houver um erro (texto vermelho), leia a mensagem — ela geralmente indica exatamente o que corrigir.

---
## Passo 1 — Conectar ao Google Drive e Instalar Software

Execute as duas células abaixo. A primeira pedirá que você **permita o acesso ao seu Google Drive** — clique no link e siga os passos.

In [ ]:
#@title 📂 Conectar ao Google Drive { display-mode: "form" }
from google.colab import drive
drive.mount('/content/drive')
print('Google Drive conectado com sucesso.')

In [ ]:
#@title 📦 Instalar pacotes { display-mode: "form" }

!pip install ai-edge-litert onnxruntime librosa soundfile huggingface_hub -q

print('\nTodos os pacotes foram instalados com sucesso.')

---
## Passo 2 — Configuração

Preencha os formulários abaixo. **Não é necessário editar nenhum código** — basta digitar ou selecionar os valores em cada formulário e executar a célula.

Execute os três formulários de cima para baixo:
1. **Configurações Gerais** — limiar de confiança, pasta de resultados
2. **Entrada de Áudio** — onde estão suas gravações, formato dos nomes de arquivo e filtro opcional de data/hora
3. **Modelo** — onde seu modelo está armazenado e como funciona

> **Dica:** Os formulários ocultam o código automaticamente. Para ver o código subjacente, clique no ícone `{ }` no canto superior direito de qualquer célula de formulário.

In [ ]:
#@title ⚙️ Configurações Gerais { display-mode: "form" }

import os

#@markdown **Pasta de resultados** — where detection results will be saved on your Google Drive.
DRIVE_RESULTS_DIR = "/content/drive/MyDrive/audio_results"  #@param {type:"string"}

#@markdown ---
#@markdown **Limiar de detecção** — minimum confidence score to save a detection (0.0–1.0).
#@markdown Menor = mais detecções (pode incluir falsos positivos). Maior = menos, porém mais confiáveis.
MIN_CONFIDENCE = 0.25  #@param {type:"slider", min:0.0, max:1.0, step:0.05}

os.makedirs(DRIVE_RESULTS_DIR, exist_ok=True)
print(f"Pasta de resultados : {DRIVE_RESULTS_DIR}")
print(f"Confiança mínima : {MIN_CONFIDENCE}")

In [ ]:
#@title ⚙️ Pré-processamento de Áudio { display-mode: "form" }
#@markdown Optional preprocessing applied to each recording before inference.
#@markdown
#@markdown ---
#@markdown **Filtro de frequência** — remove frequências fora da faixa de interesse.
FILTER_TYPE = "none"  #@param ["none", "lowpass", "highpass", "bandpass"]

#@markdown **Frequência de corte inferior (Hz)** — usada nos filtros passa-alta e passa-banda.
FILTER_LOW_HZ = 0  #@param {type:"integer"}

#@markdown **Frequência de corte superior (Hz)** — usada nos filtros passa-baixa e passa-banda.
FILTER_HIGH_HZ = 15000  #@param {type:"integer"}

#@markdown ---
#@markdown **Velocidade de reprodução** — 1.0 = normal. Abaixo de 1.0 desacelera e alonga o áudio;
#@markdown acima de 1.0 acelera e encurta. Útil para deslocar o conteúdo de frequência
#@markdown para a faixa esperada pelo modelo (ex.: 0.5x divide todas as frequências pela metade).
AUDIO_SPEED = 1.0  #@param {type:"slider", min:0.25, max:4.0, step:0.05}

_preprocess_lines = []
if FILTER_TYPE != 'none':
    if FILTER_TYPE == 'lowpass':
        _preprocess_lines.append(f'Filter   : lowpass <= {FILTER_HIGH_HZ} Hz')
    elif FILTER_TYPE == 'highpass':
        _preprocess_lines.append(f'Filter   : highpass >= {FILTER_LOW_HZ} Hz')
    elif FILTER_TYPE == 'bandpass':
        _preprocess_lines.append(f'Filter   : bandpass {FILTER_LOW_HZ}-{FILTER_HIGH_HZ} Hz')
if AUDIO_SPEED != 1.0:
    _preprocess_lines.append(f'Speed    : {AUDIO_SPEED}x')
if _preprocess_lines:
    print('Pré-processamento ativado:')
    for _l in _preprocess_lines:
        print(f'  {_l}')
else:
    print('Pré-processamento : nenhum')

#@markdown ---
#@markdown **Filtro de horário** — analisa apenas gravações dentro de uma janela de tempo diária específica.
#@markdown Deixe desativado para analisar todas as gravações independentemente do horário.
TIME_FILTER_ENABLED = False  #@param {type:"boolean"}
#@markdown **Início da janela (HH:MM)**
TIME_FILTER_START = "06:00"  #@param {type:"string"}
#@markdown **Fim da janela (HH:MM)** — pode cruzar a meia-noite, ex.: 22:00 a 06:00.
TIME_FILTER_END = "20:00"  #@param {type:"string"}

if TIME_FILTER_ENABLED:
    print(f'Filtro de horário : {TIME_FILTER_START} – {TIME_FILTER_END}')

In [ ]:
#@title 🗂️ Entrada de Áudio { display-mode: "form" }

#@markdown **Pasta de áudio** — caminho para a pasta no seu Google Drive que contém as gravações.
#@markdown - Se a pasta tiver **subpastas**, cada subpasta será tratada como um ponto de gravação separado.
#@markdown - Se todos os arquivos estiverem diretamente nesta pasta, o nome da pasta será usado como ponto.
#@markdown Exemplo: `/content/drive/MyDrive/meu_projeto/audio`
DRIVE_INPUT_DIR = "/content/drive/MyDrive/audio"  #@param {type:"string"}

#@markdown ---
#@markdown **Prefixo do nome de arquivo** — deixe em branco para nomes padrão AudioMoth (`YYYYMMDD_HHMMSS.wav`).
#@markdown Se seu gravador adiciona um prefixo antes do timestamp, insira aqui **sem** o underscore final.
#@markdown Exemplo: insira `SM4` para arquivos com nome `SM4_20250615_203000.wav`
FILENAME_PREFIX = ""  #@param {type:"string"}

#@markdown ---
#@markdown **Filtro de data/hora** (opcional) — quando ativado, apenas os arquivos cujo timestamp no nome
#@markdown esteja neste intervalo serão analisados. Arquivos sem timestamp interpretável são sempre incluídos.
#@markdown Deixe desativado para analisar todos os arquivos na pasta.
FILTER_ENABLED = False  #@param {type:"boolean"}
FILTER_START_DATE = "2025-01-01"  #@param {type:"date"}
FILTER_START_TIME = "00:00"  #@param {type:"string"}
FILTER_END_DATE = "2025-12-31"  #@param {type:"date"}
FILTER_END_TIME = "23:59"  #@param {type:"string"}

#@markdown ---
#@markdown **Coordenadas dos pontos** (opcional) — latitude e longitude por ponto de gravação.
#@markdown Formato: `NOME_PONTO=lat,lon` separados por ponto e vírgula.
#@markdown O nome do ponto deve corresponder exatamente ao nome da subpasta (ou da pasta raiz, se não houver subpastas).
#@markdown Exemplo: `PONTO_A=-15.1,-47.2;PONTO_B=-16.3,-48.1`
SITE_COORDINATES = ""  #@param {type:"string"}

FILENAME_PREFIX = FILENAME_PREFIX.strip()

_latlon_map = {}
for _e in SITE_COORDINATES.split(';'):
    _e = _e.strip()
    if '=' in _e:
        _n, _c = _e.split('=', 1)
        _p = _c.split(',')
        try: _latlon_map[_n.strip()] = (float(_p[0]), float(_p[1]))
        except Exception: pass

if not os.path.isdir(DRIVE_INPUT_DIR):
    print(f"AVISO: Pasta não encontrada: {DRIVE_INPUT_DIR}")
    print("Verifique o caminho acima — certifique-se de que o Google Drive está conectado e a pasta existe.")
else:
    print(f"Pasta de áudio : {DRIVE_INPUT_DIR}")
    if FILENAME_PREFIX:
        print(f"Prefixo do arquivo : '{FILENAME_PREFIX}'  (espera arquivos como {FILENAME_PREFIX}_YYYYMMDD_HHMMSS.wav)")
    else:
        print("Prefixo do arquivo : nenhum  (espera arquivos como YYYYMMDD_HHMMSS.wav)")
    if FILTER_ENABLED:
        print(f"Filtro de data      : {FILTER_START_DATE} {FILTER_START_TIME} → {FILTER_END_DATE} {FILTER_END_TIME}")
    else:
        print("Filtro de data      : desativado (todos os arquivos serão analisados)")
    if _latlon_map:
        for _sn, (_la, _lo) in _latlon_map.items():
            print(f"  Coordenadas   : {_sn} → lat={_la}, lon={_lo}")
    else:
        print("Coordenadas dos pontos: não definidas")

In [ ]:
#@title 🤖 Modelo { display-mode: "form" }

#@markdown **Origem do modelo** — onde está armazenado o arquivo do modelo?
MODEL_SOURCE = "huggingface"  #@param ["huggingface", "google_drive"]

#@markdown ---
#@markdown ### Se a origem do modelo = Google Drive
#@markdown Caminho completo para o arquivo do modelo no Drive (`.tflite` ou `.onnx`).
#@markdown Exemplo: `/content/drive/MyDrive/models/frog_detector.tflite`
DRIVE_MODEL_PATH  = "/content/drive/MyDrive/models/my_model.tflite"  #@param {type:"string"}
#@markdown Caminho completo para o arquivo de rótulos no Drive.
DRIVE_LABELS_PATH = "/content/drive/MyDrive/models/labels.txt"  #@param {type:"string"}

#@markdown ---
#@markdown ### Se a origem do modelo = HuggingFace
#@markdown O ID do repositório é a parte após `huggingface.co/` na URL do modelo.
#@markdown Padrão: `justinchuby/BirdNET-onnx` (BirdNET v2.4 em formato ONNX)
HF_REPO_ID     = "justinchuby/BirdNET-onnx"  #@param {type:"string"}
HF_MODEL_FILE  = "model.onnx"               #@param {type:"string"}
#@markdown O arquivo de rótulos pode estar em um repositório HuggingFace **diferente**. Deixe em branco para usar o mesmo repositório do modelo.
HF_LABELS_REPO = ""                          #@param {type:"string"}
HF_LABELS_FILE = "BirdNET_GLOBAL_6K_V2.4_Labels.txt"  #@param {type:"string"}

#@markdown ---
#@markdown ### Configurações do arquivo de rótulos
#@markdown **Possui linha de cabeçalho?** — marque se a primeira linha do arquivo de rótulos é um cabeçalho de coluna (não um rótulo).
LABELS_HAS_HEADER = False  #@param {type:"boolean"}
#@markdown **Índice da coluna de rótulo** — qual coluna contém o nome do rótulo? (0 = primeira coluna, 1 = segunda, etc.)
LABELS_COLUMN_INDEX = 0  #@param {type:"integer"}
#@markdown **Delimitador de coluna** — como as colunas são separadas no arquivo de rótulos.
LABELS_DELIMITER = "underscore (_)"  #@param ["tab", "comma (,)", "semicolon (;)", "underscore (_)"]

#@markdown ---
#@markdown **Função de ativação** — como o modelo converte sua saída em pontuações de confiança.
#@markdown `sigmoid` = cada espécie pontuada independentemente (mais comum). `softmax` = pontuações somam 1. `none` = o modelo já produz probabilidades.
ACTIVATION = "sigmoid"  #@param ["sigmoid", "softmax", "none"]

#@markdown **Taxa de amostragem (Hz)** — taxa de amostragem de áudio esperada pelo modelo.
#@markdown BirdNET = 48000 · Google Perch = 32000 · Personalizado: consulte a documentação do modelo.
SAMPLE_RATE = 48000  #@param {type:"integer"}

#@markdown **Duração do segmento (segundos)** — duração de cada trecho de áudio enviado ao modelo.
#@markdown BirdNET = 3.0 · Google Perch = 5.0 · Personalizado: consulte a documentação do modelo.
SEGMENT_DURATION_S = 3.0  #@param {type:"number"}

#@markdown **Sobreposição de segmentos (0,0–0,9)** — fração de sobreposição entre trechos consecutivos.
#@markdown 0,0 = sem sobreposição (padrão). 0,5 = 50% de sobreposição (o dobro de segmentos, maior cobertura).
SEGMENT_OVERLAP = 0.0  #@param {type:"slider", min:0.0, max:0.9, step:0.1}

print(f"Origem do modelo       : {MODEL_SOURCE}")
print(f"Ativação               : {ACTIVATION}")
print(f"Taxa de amostragem     : {SAMPLE_RATE} Hz")
print(f"Duração do segmento    : {SEGMENT_DURATION_S} s")
print(f"Sobreposição           : {SEGMENT_OVERLAP}")
print(f"Rótulos: índice={LABELS_COLUMN_INDEX}, delimitador='{LABELS_DELIMITER}', cabeçalho={LABELS_HAS_HEADER}")

---
## Passo 3 — Varrer arquivos de áudio

Esta célula varre sua pasta de áudio, descobre todos os pontos de gravação e arquivos, e exibe um resumo do que será analisado.

**Detecção de pontos:**
- Se a pasta de áudio contiver **subpastas**, cada subpasta é tratada como um ponto de gravação.
- Se todos os arquivos de áudio estiverem **diretamente na pasta raiz**, a própria pasta é tratada como um ponto único.

**Interpretação de datetime:** os timestamps são lidos dos nomes dos arquivos usando o prefixo configurado. Arquivos cujos nomes não correspondem ao padrão esperado ainda são incluídos — suas colunas de data/hora nos resultados ficarão em branco.

In [ ]:
#@title 🔍 Varrer { display-mode: "form" }

import os
import re
import glob as _glob
from collections import defaultdict
from datetime import datetime

def parse_file_datetime(filename, prefix=''):
    name = os.path.splitext(os.path.basename(filename))[0]
    m = re.search(r'(\d{4})-(\d{2})-(\d{2})_(\d{2})-(\d{2})-(\d{2})', name)
    if m:
        return datetime(int(m.group(1)), int(m.group(2)), int(m.group(3)),
                        int(m.group(4)), int(m.group(5)), int(m.group(6)))
    if prefix:
        m = re.search(rf'{re.escape(prefix)}_(\d{{8}})_(\d{{6}})', name)
    else:
        m = re.search(r'(\d{8})_(\d{6})', name)
    if m:
        d, t = m.group(1), m.group(2)
        return datetime(int(d[:4]), int(d[4:6]), int(d[6:8]),
                        int(t[:2]), int(t[2:4]), int(t[4:6]))
    return None

def get_site_name(audio_path, input_dir):
    parent = os.path.normpath(os.path.dirname(audio_path))
    root   = os.path.normpath(input_dir)
    if parent == root:
        return os.path.basename(root)
    return os.path.basename(parent)

if not os.path.isdir(DRIVE_INPUT_DIR):
    raise FileNotFoundError(
        f"Audio folder not found: {DRIVE_INPUT_DIR}\n"
        "Please check the path in the Audio Input form (Step 2)."
    )

all_audio = sorted(
    _glob.glob(os.path.join(DRIVE_INPUT_DIR, '**', '*.wav'),  recursive=True) +
    _glob.glob(os.path.join(DRIVE_INPUT_DIR, '**', '*.flac'), recursive=True) +
    _glob.glob(os.path.join(DRIVE_INPUT_DIR, '**', '*.mp3'),  recursive=True)
)

if FILTER_ENABLED and all_audio:
    _start = datetime.strptime(f"{FILTER_START_DATE} {FILTER_START_TIME or '00:00'}", '%Y-%m-%d %H:%M')
    _end   = datetime.strptime(f"{FILTER_END_DATE} {FILTER_END_TIME or '23:59'}", '%Y-%m-%d %H:%M')
    _n_before = len(all_audio)
    all_audio = [
        f for f in all_audio
        if (lambda dt: dt is None or _start <= dt <= _end)(parse_file_datetime(f, FILENAME_PREFIX))
    ]
    print(f"Date filter  : {_start.strftime('%Y-%m-%d %H:%M')} → {_end.strftime('%Y-%m-%d %H:%M')}")
    print(f"Files excluded by filter : {_n_before - len(all_audio)}")
    print()

if not all_audio:
    print(f"Nenhum arquivo de áudio encontrado em: {DRIVE_INPUT_DIR}")
    print("Formatos suportados: .wav, .flac, .mp3")
else:
    sites = defaultdict(list)
    no_datetime = []

    for f in all_audio:
        site = get_site_name(f, DRIVE_INPUT_DIR)
        dt   = parse_file_datetime(f, FILENAME_PREFIX)
        sites[site].append((f, dt))
        if dt is None:
            no_datetime.append(os.path.basename(f))

    print(f"Pasta de áudio : {DRIVE_INPUT_DIR}")
    print(f"Total de arquivos  : {len(all_audio)}")
    print(f"Pontos encontrados : {len(sites)}")
    print()

    for site_name, entries in sorted(sites.items()):
        dts = [dt for _, dt in entries if dt is not None]
        if dts:
            date_range = f"{min(dts).strftime('%Y-%m-%d %H:%M')} → {max(dts).strftime('%Y-%m-%d %H:%M')}"
        else:
            date_range = '(no timestamps parsed)'
        print(f"  {site_name}")
        print(f"    Arquivos : {len(entries)}")
        print(f"    Intervalo : {date_range}")

    if no_datetime:
        print()
        print(f"  AVISO: {len(no_datetime)} arquivo(s) não correspondem ao padrão de nome:")
        for fn in no_datetime[:5]:
            print(f"    {fn}")
        if len(no_datetime) > 5:
            print(f"    ... and {len(no_datetime) - 5} more")
        if FILENAME_PREFIX:
            print(f"  Padrão esperado: {FILENAME_PREFIX}_YYYYMMDD_HHMMSS.*")
        else:
            print(f"  Padrão esperado: YYYYMMDD_HHMMSS.*")
        print("  Esses arquivos ainda serão analisados — as colunas de data/hora nos resultados ficarão em branco.")

    print()
    print("Varredura concluída. Continue para o Passo 4.")

---
## Passo 4 — Carregar o modelo e os rótulos de espécies

Esta célula carrega seu modelo e lê a lista de espécies (ou outras classes) que ele pode detectar.

**Formato do arquivo de rótulos:** um arquivo de texto simples com um rótulo por linha, por exemplo:
```
Phylodytes luteolus
Dendropsophus minutus
Background noise
```

O número de linhas no arquivo de rótulos deve corresponder ao número de saídas do seu modelo.

In [ ]:
#@title 🧠 Carregar modelo e rótulos { display-mode: "form" }

import os
import numpy as np

_DELIMITERS = {"tab": "\t", "comma (,)": ",", "semicolon (;)":
               ";", "underscore (_)": "_"}
_labels_sep = _DELIMITERS.get(LABELS_DELIMITER, "\t")

if MODEL_SOURCE == 'huggingface':
    from huggingface_hub import hf_hub_download
    print(f'Baixando modelo do HuggingFace: {HF_REPO_ID} / {HF_MODEL_FILE} ...')
    model_path = hf_hub_download(repo_id=HF_REPO_ID, filename=HF_MODEL_FILE)
    _labels_repo = HF_LABELS_REPO.strip() or HF_REPO_ID
    print(f'Baixando rótulos do HuggingFace: {_labels_repo} / {HF_LABELS_FILE} ...')
    labels_path = hf_hub_download(repo_id=_labels_repo, filename=HF_LABELS_FILE)
elif MODEL_SOURCE == 'google_drive':
    model_path  = DRIVE_MODEL_PATH
    labels_path = DRIVE_LABELS_PATH
else:
    raise ValueError(f"MODEL_SOURCE must be 'huggingface' or 'google_drive', got: {MODEL_SOURCE!r}")

if not os.path.exists(model_path):
    raise FileNotFoundError(
        f"Model file not found: {model_path}\n"
        "Please check the path in the Model form (Step 2)."
    )
if not os.path.exists(labels_path):
    raise FileNotFoundError(
        f"Labels file not found: {labels_path}\n"
        "Please check the path in the Model form (Step 2)."
    )

ext = os.path.splitext(model_path)[1].lower()
if ext == '.tflite':
    MODEL_TYPE = 'tflite'
elif ext == '.onnx':
    MODEL_TYPE = 'onnx'
else:
    raise ValueError(f"Unsupported model format: '{ext}'.\nThe model file must end in .tflite or .onnx")

MODEL_NAME = os.path.splitext(os.path.basename(model_path))[0]

if MODEL_TYPE == 'tflite':
    from ai_edge_litert.interpreter import Interpreter as TFLiteInterpreter
    model = TFLiteInterpreter(model_path=model_path)
    model.allocate_tensors()
    in_shape  = model.get_input_details()[0]['shape']
    out_shape = model.get_output_details()[0]['shape']
    print(f'Modelo TFLite carregado.')
    print(f'  Formato de entrada : {in_shape}')
    print(f'  Formato de saída   : {out_shape}')
elif MODEL_TYPE == 'onnx':
    import onnxruntime as ort
    model = ort.InferenceSession(model_path)
    in_shape  = model.get_inputs()[0].shape
    out_shape = model.get_outputs()[0].shape
    print(f'Modelo ONNX carregado.')
    print(f'  Formato de entrada : {in_shape}')
    print(f'  Formato de saída   : {out_shape}')

labels = []
with open(labels_path, 'r', encoding='utf-8') as f:
    lines = f.readlines()

if LABELS_HAS_HEADER and lines:
    lines = lines[1:]

for line in lines:
    line = line.strip()
    if not line:
        continue
    if _labels_sep in line:
        parts = line.split(_labels_sep)
        if LABELS_COLUMN_INDEX < len(parts):
            labels.append(parts[LABELS_COLUMN_INDEX].strip())
        else:
            print(f"  AVISO: índice de coluna {LABELS_COLUMN_INDEX} não encontrado na linha: {line!r} — ignorando.")
    else:
        labels.append(line)

print(f'\nRótulos carregados: {len(labels)} classes')
print(f'  Delimitador : {LABELS_DELIMITER}  |  Índice da coluna : {LABELS_COLUMN_INDEX}  |  Cabeçalho ignorado : {LABELS_HAS_HEADER}')
print(f'  Primeiros 5 : {labels[:5]}')
print(f'  Nome do modelo: {MODEL_NAME}')

---
## Passo 5 — Verificar análises já concluídas

Esta célula verifica sua pasta de resultados para ver quais arquivos de áudio **já foram analisados**. Esses arquivos serão **ignorados** no próximo passo, para que você possa executar a análise novamente sem repetir o trabalho já feito.

In [ ]:
#@title 🔎 Verificar { display-mode: "form" }

import os
import re
import glob as _glob
from datetime import datetime

os.makedirs(DRIVE_RESULTS_DIR, exist_ok=True)

def parse_file_datetime(filename, prefix=''):
    name = os.path.splitext(os.path.basename(filename))[0]
    m = re.search(r'(\d{4})-(\d{2})-(\d{2})_(\d{2})-(\d{2})-(\d{2})', name)
    if m:
        return datetime(int(m.group(1)), int(m.group(2)), int(m.group(3)),
                        int(m.group(4)), int(m.group(5)), int(m.group(6)))
    if prefix:
        m = re.search(rf'{re.escape(prefix)}_(\d{{8}})_(\d{{6}})', name)
    else:
        m = re.search(r'(\d{8})_(\d{6})', name)
    if m:
        d, t = m.group(1), m.group(2)
        return datetime(int(d[:4]), int(d[4:6]), int(d[6:8]),
                        int(t[:2]), int(t[2:4]), int(t[4:6]))
    return None

def get_site_name(audio_path, input_dir):
    parent = os.path.normpath(os.path.dirname(audio_path))
    root   = os.path.normpath(input_dir)
    if parent == root:
        return os.path.basename(root)
    return os.path.basename(parent)

def get_result_path(audio_path, input_dir, results_dir, model_name, prefix=''):
    filename  = os.path.basename(audio_path)
    site_name = get_site_name(audio_path, input_dir)
    file_dt   = parse_file_datetime(filename, prefix)
    if file_dt is not None:
        dt_str    = file_dt.strftime('%Y%m%d_%H%M%S')
        result_fn = f"{site_name}_{dt_str}.{model_name}.results.txt"
    else:
        file_base = os.path.splitext(filename)[0]
        result_fn = f"{site_name}_{file_base}.{model_name}.results.txt"
    return os.path.join(results_dir, site_name, result_fn)

all_audio = sorted(
    _glob.glob(os.path.join(DRIVE_INPUT_DIR, '**', '*.wav'),  recursive=True) +
    _glob.glob(os.path.join(DRIVE_INPUT_DIR, '**', '*.flac'), recursive=True) +
    _glob.glob(os.path.join(DRIVE_INPUT_DIR, '**', '*.mp3'),  recursive=True)
)

if FILTER_ENABLED and all_audio:
    _start = datetime.strptime(f"{FILTER_START_DATE} {FILTER_START_TIME or '00:00'}", '%Y-%m-%d %H:%M')
    _end   = datetime.strptime(f"{FILTER_END_DATE} {FILTER_END_TIME or '23:59'}", '%Y-%m-%d %H:%M')
    all_audio = [
        f for f in all_audio
        if (lambda dt: dt is None or _start <= dt <= _end)(parse_file_datetime(f, FILENAME_PREFIX))
    ]

if TIME_FILTER_ENABLED and all_audio:
    from datetime import time as _dtime
    _ps = TIME_FILTER_START.split(':')
    _pe = TIME_FILTER_END.split(':')
    _ts = _dtime(int(_ps[0]), int(_ps[1]))
    _te = _dtime(int(_pe[0]), int(_pe[1]))
    def _tod_ok(f):
        dt = parse_file_datetime(f, FILENAME_PREFIX)
        if dt is None:
            return True
        t = dt.time()
        return (_ts <= t <= _te) if _ts <= _te else (t >= _ts or t <= _te)
    all_audio = [f for f in all_audio if _tod_ok(f)]

to_process   = [f for f in all_audio
                if not os.path.exists(
                    get_result_path(f, DRIVE_INPUT_DIR, DRIVE_RESULTS_DIR,
                                    MODEL_NAME, FILENAME_PREFIX))]
already_done = [f for f in all_audio if f not in to_process]

print(f'Pasta de resultados : {DRIVE_RESULTS_DIR}')
print(f'Nome do modelo      : {MODEL_NAME}')
if FILTER_ENABLED:
    print(f'Date filter    : {FILTER_START_DATE} {FILTER_START_TIME} → {FILTER_END_DATE} {FILTER_END_TIME}')
print()
print(f'Total de arquivos   : {len(all_audio)}')
print(f'Já analisados       : {len(already_done)}')
print(f'Restando analisar   : {len(to_process)}')

if already_done:
    print()
    print('Já concluídos (serão ignorados):')
    for f in already_done:
        rp = get_result_path(f, DRIVE_INPUT_DIR, DRIVE_RESULTS_DIR,
                             MODEL_NAME, FILENAME_PREFIX)
        print(f"  {os.path.basename(f)}  →  {os.path.basename(rp)}")

if not to_process:
    print()
    print('Todos os arquivos já foram analisados. Nada a fazer!')
    print('Para re-analisar, exclua primeiro os arquivos de resultados correspondentes do seu Drive.')
else:
    print()
    print(f'Pronto para analisar {len(to_process)} arquivo(s). Prossiga para o Passo 6.')

---
## Passo 6 — Executar a análise e salvar resultados

Este é o passo principal de análise. Para cada arquivo de áudio:
1. A gravação é dividida em segmentos curtos (ex.: 3 segundos cada)
2. Cada segmento é enviado ao modelo
3. Detecções acima do limiar de confiança são mantidas
4. Os resultados são salvos imediatamente no seu Google Drive

**Arquivos de resultado** são salvos como:
```
DRIVE_RESULTS_DIR / NOME_DO_PONTO / arquivo_audio.NOME_DO_MODELO.results.txt
```

Cada arquivo de resultados tem as colunas: `site`, `date`, `time`, `start_time`, `end_time`, `label`, `confidence`

- `site` — o nome do ponto de gravação (nome da subpasta, ou da pasta raiz se não houver subpastas).
- `date` / `time` — timestamp da detecção, lido do nome do arquivo.
- `start_time` / `end_time` — deslocamento em segundos desde o início do arquivo de áudio.

> Dependendo do número de arquivos e da duração das gravações, este passo pode demorar. O progresso é mostrado abaixo.

In [ ]:
#@title 🚀 Executar { display-mode: "form" }
import csv
import re
import shutil
import time
import numpy as np
import librosa
from datetime import datetime, timedelta

def parse_file_datetime(filename, prefix=''):
    name = os.path.splitext(os.path.basename(filename))[0]
    m = re.search(r'(\d{4})-(\d{2})-(\d{2})_(\d{2})-(\d{2})-(\d{2})', name)
    if m:
        return datetime(int(m.group(1)), int(m.group(2)), int(m.group(3)),
                        int(m.group(4)), int(m.group(5)), int(m.group(6)))
    if prefix:
        m = re.search(rf'{re.escape(prefix)}_(\d{{8}})_(\d{{6}})', name)
    else:
        m = re.search(r'(\d{8})_(\d{6})', name)
    if m:
        d, t = m.group(1), m.group(2)
        return datetime(int(d[:4]), int(d[4:6]), int(d[6:8]),
                        int(t[:2]), int(t[2:4]), int(t[4:6]))
    return None

def get_site_name(audio_path, input_dir):
    parent = os.path.normpath(os.path.dirname(audio_path))
    root   = os.path.normpath(input_dir)
    if parent == root:
        return os.path.basename(root)
    return os.path.basename(parent)

def apply_activation(logits, activation):
    arr = np.array(logits, dtype=np.float32).flatten()
    if activation == 'sigmoid':
        return 1.0 / (1.0 + np.exp(-arr))
    elif activation == 'softmax':
        arr = arr - arr.max()
        e = np.exp(arr)
        return e / e.sum()
    return arr

def run_model(model, model_type, audio_segment):
    seg = audio_segment.astype(np.float32)
    if model_type == 'tflite':
        in_det  = model.get_input_details()[0]
        out_det = model.get_output_details()[0]
        try:
            model.set_tensor(in_det['index'], seg.reshape(in_det['shape']))
        except ValueError:
            model.set_tensor(in_det['index'], seg.reshape(1, -1))
        model.invoke()
        return model.get_tensor(out_det['index']).flatten()
    else:
        input_name = model.get_inputs()[0].name
        try:
            logits = model.run(None, {input_name: seg.reshape(1, -1)})[0]
        except Exception:
            logits = model.run(None, {input_name: seg[np.newaxis, :]})[0]
        return np.array(logits).flatten()

def preprocess_audio(audio, sr):
    if FILTER_TYPE != 'none':
        from scipy.signal import butter, sosfilt
        nyq = sr / 2.0
        if FILTER_TYPE == 'lowpass':
            sos = butter(5, min(FILTER_HIGH_HZ, nyq - 1) / nyq, btype='low', output='sos')
        elif FILTER_TYPE == 'highpass':
            sos = butter(5, max(FILTER_LOW_HZ, 1) / nyq, btype='high', output='sos')
        elif FILTER_TYPE == 'bandpass':
            lo = max(FILTER_LOW_HZ, 1) / nyq
            hi = min(FILTER_HIGH_HZ, nyq - 1) / nyq
            sos = butter(5, [lo, hi], btype='band', output='sos')
        audio = sosfilt(sos, audio).astype(np.float32)
    if AUDIO_SPEED != 1.0:
        audio = librosa.effects.time_stretch(audio, rate=AUDIO_SPEED)
    return audio

segment_samples  = int(SEGMENT_DURATION_S * SAMPLE_RATE)
stride_samples   = max(1, int(segment_samples * (1 - SEGMENT_OVERLAP)))
def _in_time_window(file_dt, start_str, end_str):
    from datetime import time as _dtime
    if file_dt is None:
        return True
    t = file_dt.time()
    ps, pe = start_str.split(':'), end_str.split(':')
    t_start = _dtime(int(ps[0]), int(ps[1]))
    t_end   = _dtime(int(pe[0]), int(pe[1]))
    if t_start <= t_end:
        return t_start <= t <= t_end
    return t >= t_start or t <= t_end

total_detections = 0
total_start      = time.time()

if not to_process:
    print('Nenhum arquivo para processar. Todas as análises já estão concluídas.')
else:
    print(f'Iniciando análise de {len(to_process)} arquivo(s) com o modelo "{MODEL_NAME}"')
    print(f'Limiar de confiança  : {MIN_CONFIDENCE}')
    print(f'Duração do segmento  : {SEGMENT_DURATION_S}s  |  Taxa de amostragem: {SAMPLE_RATE} Hz')
    if FILTER_TYPE != 'none' or AUDIO_SPEED != 1.0:
        _pp = []
        if FILTER_TYPE != 'none': _pp.append(f'filter={FILTER_TYPE}')
        if AUDIO_SPEED != 1.0:   _pp.append(f'speed={AUDIO_SPEED}x')
        print(f'Preprocessing        : {", ".join(_pp)}')
    print('=' * 65)

    for file_idx, audio_path in enumerate(to_process, 1):
        filename  = os.path.basename(audio_path)
        site_name = get_site_name(audio_path, DRIVE_INPUT_DIR)
        file_dt   = parse_file_datetime(filename, FILENAME_PREFIX)
        if TIME_FILTER_ENABLED and not _in_time_window(file_dt, TIME_FILTER_START, TIME_FILTER_END):
            print(f'[{file_idx}/{len(to_process)}] {filename}  — ignorado (fora de {TIME_FILTER_START}–{TIME_FILTER_END})')
            continue

        if file_dt is not None:
            dt_str    = file_dt.strftime('%Y%m%d_%H%M%S')
            result_fn = f"{site_name}_{dt_str}.{MODEL_NAME}.results.txt"
        else:
            file_base = os.path.splitext(filename)[0]
            result_fn = f"{site_name}_{file_base}.{MODEL_NAME}.results.txt"
            print(f"  AVISO: não foi possível interpretar data/hora de '{filename}' — usando nome do arquivo.")

        print(f'[{file_idx}/{len(to_process)}] {filename}  (ponto: {site_name})')

        os.makedirs('/content/audio_tmp', exist_ok=True)
        _local = os.path.join('/content/audio_tmp', os.path.basename(audio_path))
        shutil.copy2(audio_path, _local)
        try:
            audio, _ = librosa.load(_local, sr=SAMPLE_RATE, mono=True)
        except Exception as e:
            os.remove(_local)
            print(f'  ERRO ao carregar áudio: {e} — ignorando.')
            continue
        os.remove(_local)

        audio = preprocess_audio(audio, SAMPLE_RATE)

        rows       = []
        file_start = time.time()
        n_segments = 0

        for start_samp in range(0, len(audio), stride_samples):
            seg = audio[start_samp:start_samp + segment_samples]
            if len(seg) < segment_samples * 0.5:
                continue
            if len(seg) < segment_samples:
                seg = np.pad(seg, (0, segment_samples - len(seg)))

            try:
                logits = run_model(model, MODEL_TYPE, seg)
            except Exception as e:
                print(f'  ERRO durante inferência em {start_samp/SAMPLE_RATE:.1f}s: {e}')
                continue

            n_segments += 1
            scores     = apply_activation(logits, ACTIVATION)
            offset_s   = start_samp / SAMPLE_RATE
            end_s      = offset_s + SEGMENT_DURATION_S

            for idx, score in enumerate(scores):
                if float(score) >= MIN_CONFIDENCE and idx < len(labels):
                    if file_dt is not None:
                        date_str = file_dt.strftime('%Y-%m-%d')
                        time_str = file_dt.strftime('%H:%M:%S')
                    else:
                        date_str = ''
                        time_str = ''
                    site_lat, site_lon = _latlon_map.get(site_name, ('', ''))
                    rows.append([site_name, site_lat, site_lon, date_str, time_str,
                                 f'{offset_s:.1f}', f'{end_s:.1f}',
                                 labels[idx], f'{float(score):.4f}'])

        elapsed     = time.time() - file_start
        per_seg     = elapsed / n_segments if n_segments else 0
        audio_dur_s = len(audio) / SAMPLE_RATE

        site_dir    = os.path.join(DRIVE_RESULTS_DIR, site_name)
        os.makedirs(site_dir, exist_ok=True)
        result_path = os.path.join(site_dir, result_fn)

        with open(result_path, 'w', newline='', encoding='utf-8') as f:
            writer = csv.writer(f)
            writer.writerow(['site', 'lat', 'lon', 'date', 'time', 'start_time', 'end_time', 'label', 'confidence'])
            writer.writerows(rows)

        total_detections += len(rows)
        print(f'  -> {len(rows)} detection(s)  |  {n_segments} segments  |  '
              f'{elapsed:.1f}s ({per_seg*1000:.0f}ms/seg, {audio_dur_s/elapsed:.1f}x realtime)  '
              f'→  {result_fn}')

    total_elapsed = time.time() - total_start
    print()
    print('=' * 65)
    print(f'Analysis complete!')
    print(f'  Arquivos analisados  : {len(to_process)}')
    print(f'  Total de detecções   : {total_detections}')
    print(f'  Tempo total          : {total_elapsed:.1f}s')
    print(f'  Resultados salvos em : {DRIVE_RESULTS_DIR}')